# Classification — Random Linear Classifier (training data only)

**ML Teach by Doing · Day 08 · companion notebook 1 of 2**

What this notebook does, top to bottom:

1. Manufactures a cats-vs-dogs dataset from bell curves
2. Implements the Random Linear Classifier from Days 6–7
3. Runs it and plots the decision boundary it found

**How to use it:** run every cell in order (`Shift+Enter`, or *Run All*). Then come back and start breaking things — the experiments cell at the bottom has suggestions.

> ⚠️ Run cells **top to bottom**. The random seed only guarantees identical results if the *sequence* of random draws is identical. Re-running a middle cell twice changes everything after it.

In [ ]:
# ============================================================
# SETUP — the two libraries this notebook needs
# ============================================================
import numpy as np                # scientific computing: arrays, dot products, randomness
import matplotlib.pyplot as plt   # plotting

In [ ]:
# Same seed -> same "random" numbers every run -> same results as the study doc.
# Try changing 42 to anything else and re-running the whole notebook.
np.random.seed(42)

## Step 1 — Gather (manufacture) the data

No animals were measured. We *sample* plausible measurements from bell curves with `np.random.normal(mean, std, size)`:

| | whisker length (x1) | ear flappiness (x2) |
|---|---|---|
| **dogs** | around **5** | around **8** |
| **cats** | around **8** | around **5** |

Standard deviation 1 everywhere → almost all values land within ±2 of the mean.

In [ ]:
# ============================================================
# STEP 1 — manufacture the data from bell curves
# ============================================================
N = 10  # animals per species

dog_whisker_length = np.random.normal(5, 1, N)   # (10,) — values near 5
dog_ear_flappiness = np.random.normal(8, 1, N)   # (10,) — values near 8
cat_whisker_length = np.random.normal(8, 1, N)   # (10,) — cats: longer whiskers
cat_ear_flappiness = np.random.normal(5, 1, N)   # (10,) — cats: stiffer ears

print("dog_whisker_length:", np.round(dog_whisker_length, 2))
print("dog_ear_flappiness:", np.round(dog_ear_flappiness, 2))

In [ ]:
# Always LOOK at your data before doing anything clever with it.
plt.scatter(dog_whisker_length, dog_ear_flappiness, label="dogs")
plt.scatter(cat_whisker_length, cat_ear_flappiness, label="cats")
plt.xlabel("whisker length  (x1)")
plt.ylabel("ear flappiness index  (x2)")
plt.title("The cats-vs-dogs dataset")
plt.legend()
plt.show()

## Steps 2–4 — the hypothesis, the loss, the algorithm

Everything from the whiteboard, in two functions:

- **Hypothesis:** `h(x) = sign(theta · x + theta_0)` — positive score says dog (+1), negative says cat (−1)
- **Loss:** count the training mistakes
- **Algorithm (Random Linear Classifier):** try `k` random lines, keep the champion

The whiteboard formula `theta_1*x1 + theta_2*x2` and the code `np.dot(theta, x)` are the *same thing* — a dot product.

In [ ]:
# ============================================================
# STEPS 2-4 — hypothesis + loss + the Random Linear Classifier
# ============================================================
def compute_error(dogs, cats, theta, theta_0):
    # dogs, cats: (2, n) — each COLUMN is one animal
    dog_scores = np.dot(theta, dogs) + theta_0   # (n,) — theta·x + theta_0, all dogs at once
    cat_scores = np.dot(theta, cats) + theta_0   # (n,)
    # dogs should be +1 -> a negative score is a mistake
    # cats should be -1 -> a positive score is a mistake
    return np.sum(dog_scores < 0) + np.sum(cat_scores > 0)


def random_linear_classifier(dogs, cats, k, d):
    # k = how many random lines to try (hyperparameter)
    # d = number of features (2: whisker length, ear flappiness)
    best_error, best_theta, best_theta_0 = np.inf, None, None
    for _ in range(k):                        # try k random lines...
        theta   = np.random.normal(size=d)    # (2,) — random tilt
        theta_0 = np.random.normal()          # scalar — random shift
        error = compute_error(dogs, cats, theta, theta_0)
        if error < best_error:                # ...keep the champion
            best_error, best_theta, best_theta_0 = error, theta, theta_0
    return best_theta, best_theta_0, best_error

## Pre-processing — ten numbers become one matrix

`np.vstack` stacks the two measurement arrays into a `(2, 10)` matrix, so that **column j = animal j** = the point `(x1, x2)`. That layout is exactly what `np.dot(theta, data)` needs to score all ten animals in one line.

In [ ]:
# ============================================================
# PRE-PROCESSING — pack each species into a (2, n) matrix
# ============================================================
dogs_data = np.vstack((dog_whisker_length, dog_ear_flappiness))  # (2, 10)
cats_data = np.vstack((cat_whisker_length, cat_ear_flappiness))  # (2, 10)

print("dogs_data shape:", dogs_data.shape)
print("cats_data shape:", cats_data.shape)
assert dogs_data.shape == (2, N)   # sanity check before trusting any dot products

## Step 5 — run it

In [ ]:
# ============================================================
# STEP 5 — press go
# ============================================================
best_theta, best_theta_0, best_error = random_linear_classifier(dogs_data, cats_data, k=100, d=2)

print("best_theta:   ", np.round(best_theta, 2))        # (theta_1, theta_2) — the tilt
print("best_theta_0: ", round(float(best_theta_0), 2))  # the shift
print("training error:", best_error, "mistakes out of", 2 * N)

**Expected with seed 42:** `best_theta ≈ [-0.68, 0.61]`, `best_theta_0 ≈ 1.03`, training error `0`.

(The lecture got `[-0.68, 0.66]` and `-0.46` — a different seed and draw order give a different champion. What matters is not the numbers but whether the line separates the classes.)

In [ ]:
# ============================================================
# SEE the answer — plot the decision boundary
# ============================================================
# The boundary is the line   theta_1*x1 + theta_2*x2 + theta_0 = 0.
# Solve for x2 so we can plot it:   x2 = -(theta_1*x1 + theta_0) / theta_2
x1 = np.linspace(2, 11, 100)
x2 = -(best_theta[0] * x1 + best_theta_0) / best_theta[1]

plt.scatter(dog_whisker_length, dog_ear_flappiness, label="dogs (+1)")
plt.scatter(cat_whisker_length, cat_ear_flappiness, label="cats (-1)")
plt.plot(x1, x2, "r-", label="decision boundary")
plt.xlabel("whisker length  (x1)")
plt.ylabel("ear flappiness index  (x2)")
plt.title("What the Random Linear Classifier found (k=100)")
plt.ylim(2, 11)
plt.legend()
plt.show()

## Experiments — break it on purpose

The fastest way to learn is to change one thing, predict what will happen, then run it.

1. **Change the seed** to 7. Do you still get 0 errors? Does the line move?
2. **Set `k=1`.** One blind guess. Run the last three cells a few times — watch the boundary flail.
3. **Set `k=100000`.** Better than `k=100`? By how much? Worth the wait?
4. **Move the clusters together:** make cats `np.random.normal(6.5, 1, N)` for both features. Can *any* straight line get 0 errors now?
5. **More data:** `N = 100`. Does the boundary get more sensible?
6. **Check the scores yourself:** print `np.dot(best_theta, dogs_data) + best_theta_0` — every value should be positive. Are they?